In [1]:
import pandas as pd

# Define industrial thresholds
LIMITS = {
    'Urea_Biuret_Pct': {'type': 'max', 'val': 1.00, 'label': 'Biuret Content too High'},
    'Urea_Moisture_Pct': {'type': 'max', 'val': 0.30, 'label': 'Moisture Content too High'},
    'Urea_Prill_Size_Pct': {'type': 'min', 'val': 90.00, 'label': 'Inadequate Prill Size'},
    'Ammonia_Purity_Pct': {'type': 'min', 'val': 99.50, 'label': 'Low Ammonia Purity'},
    'Boiler_Water_pH': {'type': 'range', 'min': 8.5, 'max': 9.5, 'label': 'Boiler pH Out of Range'}
}

def analyze_lab_data(file_path):
    df = pd.read_csv(file_path)
    oos_records = []

    for index, row in df.iterrows():
        failures = []
        
        # Check Biuret
        if row['Urea_Biuret_Pct'] > LIMITS['Urea_Biuret_Pct']['val']:
            failures.append(f"Biuret ({row['Urea_Biuret_Pct']}% > {LIMITS['Urea_Biuret_Pct']['val']}%)")
            
        # Check Moisture
        if row['Urea_Moisture_Pct'] > LIMITS['Urea_Moisture_Pct']['val']:
            failures.append(f"Moisture ({row['Urea_Moisture_Pct']}% > {LIMITS['Urea_Moisture_Pct']['val']}%)")
            
        # Check Prill Size
        if row['Urea_Prill_Size_Pct'] < LIMITS['Urea_Prill_Size_Pct']['val']:
            failures.append(f"Prill Size ({row['Urea_Prill_Size_Pct']}% < {LIMITS['Urea_Prill_Size_Pct']['val']}%)")
            
        # Check Ammonia
        if row['Ammonia_Purity_Pct'] < LIMITS['Ammonia_Purity_Pct']['val']:
            failures.append(f"Ammonia Purity ({row['Ammonia_Purity_Pct']}% < {LIMITS['Ammonia_Purity_Pct']['val']}%)")
            
        # Check Boiler pH
        if row['Boiler_Water_pH'] < LIMITS['Boiler_Water_pH']['min'] or row['Boiler_Water_pH'] > LIMITS['Boiler_Water_pH']['max']:
            failures.append(f"Boiler pH ({row['Boiler_Water_pH']} is outside {LIMITS['Boiler_Water_pH']['min']}-{LIMITS['Boiler_Water_pH']['max']})")
            
        if failures:
            oos_records.append({
                'Date': row['Timestamp'],
                'Shift': row['Shift'],
                'Analyst': row['Analyst_ID'],
                'Violations': ", ".join(failures),
                'Total_Violations': len(failures)
            })
            
    oos_df = pd.DataFrame(oos_records)
    return df, oos_df

# Run the analyzer
raw_data, oos_data = analyze_lab_data('fertilizer_lab_raw_data.csv')

# Output overall stats
total_batches = len(raw_data)
failed_batches = len(oos_data)
pass_rate = ((total_batches - failed_batches) / total_batches) * 100

print(f"--- FFC/Engro Style Lab Quality Audit ---")
print(f"Total Batches Analyzed: {total_batches}")
print(f"Total Out-Of-Specification (OOS) Batches: {failed_batches}")
print(f"Overall Plant Pass Rate: {pass_rate:.2f}%")

# Save the anomaly report
oos_data.to_csv('oos_quality_report.csv', index=False)
print("Saved anomalous entries to 'oos_quality_report.csv'")

--- FFC/Engro Style Lab Quality Audit ---
Total Batches Analyzed: 1095
Total Out-Of-Specification (OOS) Batches: 200
Overall Plant Pass Rate: 81.74%
Saved anomalous entries to 'oos_quality_report.csv'
